# Introduction to Scikit-Learn

Scikit-Learn (sklearn) is the most widely used machine learning library in Python.
This notebook covers its design philosophy, the Estimator API, data representation
conventions, and walks through complete classification and regression examples.

## 1. The Scikit-Learn Ecosystem

Scikit-Learn provides a **consistent, unified API** for a wide range of ML tasks:

| Category | Examples |
|----------|----------|
| Classification | KNN, SVM, Decision Trees, Random Forests, Logistic Regression |
| Regression | Linear Regression, Ridge, Lasso, SVR |
| Clustering | KMeans, DBSCAN, Hierarchical |
| Dimensionality Reduction | PCA, t-SNE, LDA |
| Model Selection | Cross-validation, Grid Search, Train/Test Split |
| Preprocessing | Scaling, Encoding, Imputation |

### Design Principles

1. **Consistency** -- All estimators share the same interface (`fit`, `predict`, `transform`).
2. **Inspection** -- All parameters and learned attributes are public.
3. **Non-proliferation of classes** -- Data is represented as NumPy arrays or Pandas DataFrames.
4. **Composition** -- Estimators can be chained into Pipelines.
5. **Sensible defaults** -- Models work out of the box with reasonable hyperparameters.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version:       {np.__version__}")
print(f"Pandas version:      {pd.__version__}")

---
## 2. The Estimator API

Every model in sklearn follows the **Estimator API**. Understanding these three
methods is the key to using the entire library:

| Method | Purpose | Used by |
|--------|---------|----------|
| `fit(X, y)` | Learn from training data | All estimators |
| `predict(X)` | Make predictions on new data | Classifiers, Regressors |
| `transform(X)` | Transform data | Preprocessors, Dimensionality Reducers |

```python
# The universal sklearn pattern:
from sklearn.some_module import SomeModel

model = SomeModel(hyperparameter=value)   # 1. Instantiate
model.fit(X_train, y_train)               # 2. Fit on training data
predictions = model.predict(X_test)       # 3. Predict on new data
```

This pattern is **the same** whether you are using KNN, Linear Regression,
Random Forests, or Support Vector Machines.

### Learned Attributes

After calling `fit()`, the model stores what it learned as attributes ending
with an underscore (`_`). This convention makes it easy to distinguish learned
attributes from user-set hyperparameters.

```python
model.coef_          # Learned coefficients (e.g., LinearRegression)
model.intercept_     # Learned intercept
model.classes_       # Discovered classes (classifiers)
```

---
## 3. Data Representation in Scikit-Learn

Scikit-Learn expects data in a specific format:

**Features matrix `X`**: a 2D array of shape `(n_samples, n_features)`
- Each row is one observation (sample)
- Each column is one measurement (feature)

**Target vector `y`**: a 1D array of shape `(n_samples,)`
- Contains the label or value we want to predict

```
X = [[feature_1, feature_2, ..., feature_p],     # sample 1
     [feature_1, feature_2, ..., feature_p],     # sample 2
     ...                                          # ...
     [feature_1, feature_2, ..., feature_p]]     # sample n

y = [target_1, target_2, ..., target_n]
```

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data    # Features matrix
y = iris.target  # Target vector

print(f"Features matrix X:")
print(f"  Type:  {type(X)}")
print(f"  Shape: {X.shape}  -->  {X.shape[0]} samples, {X.shape[1]} features")
print(f"  Feature names: {iris.feature_names}")
print(f"\nTarget vector y:")
print(f"  Type:  {type(y)}")
print(f"  Shape: {y.shape}")
print(f"  Classes: {iris.target_names}")
print(f"  Distribution: {np.bincount(y)}")

In [ ]:
# Viewing the data as a DataFrame is often more intuitive
df = pd.DataFrame(X, columns=iris.feature_names)
df["target"] = y
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df.head(10)

---
## 4. Train/Test Split

We must evaluate our model on data it has **never seen during training**.
`train_test_split` handles this with a single function call.

Key parameters:
- `test_size`: fraction of data to hold out (default 0.25)
- `random_state`: seed for reproducibility
- `stratify`: ensures class proportions are preserved in both sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # preserve class balance
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nClass distribution in training set: {np.bincount(y_train)}")
print(f"Class distribution in test set:     {np.bincount(y_test)}")

---
## 5. Complete Example: Iris Classification with KNN

**K-Nearest Neighbors (KNN)** is one of the simplest ML algorithms:

1. Given a new data point, find the **k** closest training samples.
2. The predicted class is the **majority vote** among those k neighbors.

Mathematically, the distance between two points $\mathbf{a}$ and $\mathbf{b}$
is typically measured using **Euclidean distance**:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{p} (a_i - b_i)^2}$$

where $p$ is the number of features.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Instantiate the model
knn = KNeighborsClassifier(n_neighbors=5)

# Step 2: Fit on training data
knn.fit(X_train, y_train)

print("Model fitted. Learned attributes:")
print(f"  Classes: {knn.classes_}")
print(f"  Number of training samples: {knn.n_samples_fit_}")

In [ ]:
# Step 3: Predict on test data
y_pred = knn.predict(X_test)

# Compare predictions to actual values
comparison = pd.DataFrame({
    "Actual": [iris.target_names[i] for i in y_test],
    "Predicted": [iris.target_names[i] for i in y_pred],
})
comparison["Correct"] = comparison["Actual"] == comparison["Predicted"]
print(comparison.to_string(index=True))

In [ ]:
# Step 4: Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.2%}")
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Choosing k: The Effect of Hyperparameters

The value of `k` (number of neighbors) is a **hyperparameter** -- it is set by the
user, not learned from data. Let us see how different values affect performance.

In [ ]:
k_range = range(1, 26)
train_accuracies = []
test_accuracies = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_accuracies.append(accuracy_score(y_train, knn.predict(X_train)))
    test_accuracies.append(accuracy_score(y_test, knn.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(k_range, train_accuracies, "o-", label="Training", markersize=4)
plt.plot(k_range, test_accuracies, "s-", label="Test", markersize=4)
plt.xlabel("k (number of neighbors)", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("KNN: Effect of k on Accuracy", fontsize=14)
plt.legend(fontsize=11)
plt.xticks(range(1, 26, 2))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Complete Example: Linear Regression on Synthetic Data

**Linear Regression** assumes the target is a linear combination of the features:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_p x_p + b$$

or in vector notation:

$$\hat{y} = \mathbf{w}^T \mathbf{x} + b$$

The model finds $\mathbf{w}$ (weights/coefficients) and $b$ (intercept/bias) by
minimizing the **Mean Squared Error** (MSE) on the training data:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
from sklearn.datasets import make_regression

# Generate synthetic data: y = 3*x + 7 + noise
np.random.seed(42)
X_reg, y_reg, true_coef = make_regression(
    n_samples=100,
    n_features=1,
    noise=15,
    coef=True,
    random_state=42
)

# Add an offset to make it more interesting
y_reg = y_reg + 50

plt.figure(figsize=(8, 5))
plt.scatter(X_reg, y_reg, alpha=0.6, edgecolors="k")
plt.xlabel("Feature (X)", fontsize=12)
plt.ylabel("Target (y)", fontsize=12)
plt.title("Synthetic Regression Data", fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"True coefficient: {true_coef:.2f}")
print(f"Data shape: X={X_reg.shape}, y={y_reg.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Fit
lr = LinearRegression()
lr.fit(X_train_r, y_train_r)

print(f"Learned coefficient (w): {lr.coef_[0]:.4f}")
print(f"Learned intercept  (b): {lr.intercept_:.4f}")
print(f"\nThe model learned:  y = {lr.coef_[0]:.2f} * x + {lr.intercept_:.2f}")

In [ ]:
# Predict
y_pred_r = lr.predict(X_test_r)

# Evaluate
mse = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_r, y_pred_r)

print(f"Mean Squared Error (MSE):  {mse:.2f}")
print(f"Root Mean Squared Error:   {rmse:.2f}")
print(f"R-squared (R²):            {r2:.4f}")

In [ ]:
# Visualize the fit
plt.figure(figsize=(8, 5))
plt.scatter(X_train_r, y_train_r, alpha=0.5, label="Training data", edgecolors="k")
plt.scatter(X_test_r, y_test_r, alpha=0.7, color="orange", label="Test data",
            edgecolors="k")

# Plot the regression line
x_line = np.linspace(X_reg.min(), X_reg.max(), 100).reshape(-1, 1)
y_line = lr.predict(x_line)
plt.plot(x_line, y_line, "r-", linewidth=2, label="Fitted line")

plt.xlabel("Feature (X)", fontsize=12)
plt.ylabel("Target (y)", fontsize=12)
plt.title(f"Linear Regression (R² = {r2:.3f})", fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Understanding R-squared ($R^2$)

$R^2$ measures how much of the variance in $y$ is explained by the model:

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- $R^2 = 1.0$ means the model explains all variance (perfect fit)
- $R^2 = 0.0$ means the model is no better than predicting the mean
- $R^2 < 0.0$ means the model is worse than predicting the mean

---
## 7. Model Evaluation Basics

Choosing the right metric depends on the type of problem.

### 7.1 Classification Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Using our earlier KNN predictions on Iris
knn_best = KNeighborsClassifier(n_neighbors=5)
knn_best.fit(X_train, y_train)
y_pred_iris = knn_best.predict(X_test)

print("=== Classification Metrics ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_iris):.4f}")
print(f"\nPer-class detail:")
print(classification_report(y_test, y_pred_iris, target_names=iris.target_names))

In [ ]:
import seaborn as sns

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_iris)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Confusion Matrix: KNN on Iris", fontsize=14)
plt.tight_layout()
plt.show()

### 7.2 Regression Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Using our linear regression predictions
y_pred_lr = lr.predict(X_test_r)

mae = mean_absolute_error(y_test_r, y_pred_lr)
mse = mean_squared_error(y_test_r, y_pred_lr)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_r, y_pred_lr)

print("=== Regression Metrics ===")
print(f"Mean Absolute Error (MAE):    {mae:.4f}")
print(f"Mean Squared Error (MSE):     {mse:.4f}")
print(f"Root Mean Squared Error:      {rmse:.4f}")
print(f"R² Score:                     {r2:.4f}")

print("\n--- Interpretation ---")
print(f"MAE: On average, predictions are off by {mae:.1f} units")
print(f"R²:  The model explains {r2:.1%} of the variance in y")

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| MAE | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error; easy to interpret |
| MSE | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Penalizes large errors more heavily |
| RMSE | $\sqrt{\text{MSE}}$ | Same units as target; most commonly reported |
| R² | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained (0 to 1 ideal) |

---
## 8. Cross-Validation

A single train/test split can be **unreliable** -- the result depends on which
samples happen to land in each set. **Cross-validation** solves this by training
and evaluating the model multiple times on different splits.

### k-Fold Cross-Validation

1. Split the data into **k** equally-sized folds.
2. For each fold:
   - Use that fold as the test set
   - Train on the remaining k-1 folds
3. Average the k test scores.

```
5-Fold Cross-Validation:

Fold 1: [TEST] [Train] [Train] [Train] [Train]
Fold 2: [Train] [TEST] [Train] [Train] [Train]
Fold 3: [Train] [Train] [TEST] [Train] [Train]
Fold 4: [Train] [Train] [Train] [TEST] [Train]
Fold 5: [Train] [Train] [Train] [Train] [TEST]
```

This gives a more **robust** estimate of model performance.

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validate the KNN classifier
knn_cv = KNeighborsClassifier(n_neighbors=5)

# 5-fold cross-validation, scoring by accuracy
cv_scores = cross_val_score(knn_cv, X, y, cv=5, scoring="accuracy")

print(f"Cross-validation scores (5 folds): {cv_scores}")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")
print(f"\nWe can report: {cv_scores.mean():.2%} +/- {cv_scores.std():.2%}")

In [ ]:
# Cross-validate the linear regression model
lr_cv = LinearRegression()

# Use negative MSE because sklearn convention: higher is better
cv_mse = cross_val_score(lr_cv, X_reg, y_reg, cv=5, scoring="neg_mean_squared_error")
cv_r2 = cross_val_score(lr_cv, X_reg, y_reg, cv=5, scoring="r2")

print(f"Cross-validated MSE:  {-cv_mse.mean():.4f} +/- {cv_mse.std():.4f}")
print(f"Cross-validated RMSE: {np.sqrt(-cv_mse.mean()):.4f}")
print(f"Cross-validated R²:   {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")

### Why Negative MSE?

Scikit-Learn's `cross_val_score` uses the convention that **higher scores are better**.
Since MSE is a loss (lower is better), sklearn negates it. To get the actual MSE,
simply negate the returned value: `mse = -score`.

In [ ]:
# Comparing different values of k using cross-validation
k_range = range(1, 26)
cv_means = []
cv_stds = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X, y, cv=10, scoring="accuracy")
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)

plt.figure(figsize=(8, 5))
plt.plot(k_range, cv_means, "o-", markersize=4)
plt.fill_between(k_range, cv_means - cv_stds, cv_means + cv_stds, alpha=0.2)
plt.xlabel("k (number of neighbors)", fontsize=12)
plt.ylabel("Cross-validated Accuracy", fontsize=12)
plt.title("KNN: Selecting k with 10-Fold Cross-Validation", fontsize=14)
plt.xticks(range(1, 26, 2))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(cv_means)]
print(f"Best k: {best_k} with accuracy {cv_means.max():.4f}")

---
## 9. Key Takeaways

1. **Scikit-Learn** provides a consistent API: `fit()`, `predict()`, `transform()`.

2. Data is represented as a **features matrix X** (n_samples, n_features) and a
   **target vector y** (n_samples,).

3. Always **split data** into training and test sets before fitting a model.

4. **Classification** predicts discrete categories; **regression** predicts
   continuous values. Each has its own evaluation metrics.

5. **Cross-validation** provides a more robust performance estimate than a single
   train/test split.

6. The same API pattern (`instantiate` -> `fit` -> `predict`) applies to **every**
   model in sklearn, making it easy to swap algorithms.

In the next notebook, we will take a deeper dive into **model evaluation metrics**
and learn when accuracy alone is not enough.